<a href="https://colab.research.google.com/github/Daria-Lytvynenko/ML_course/blob/main/fake_news_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project Description

Fake news detection using natural language processing methods.

Such models can be useful for social networks, news aggregators and platforms that want to identify false information.

# Data

- `title` - the title of the article
- `text` - the full text of the article
- `date` - the date the article was published
- `is_fake` - whether the article is fake (target column)

## 0. Dependencies

In [ ]:
!pip install nltk transformers torch --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import torch
import nltk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    classification_report, roc_auc_score
)
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
from transformers import BertTokenizer, BertModel

pio.renderers.default = 'colab'

for resource in ['punkt', 'punkt_tab', 'averaged_perceptron_tagger_eng',
                 'wordnet', 'stopwords']:
    nltk.download(resource, quiet=True)

## 1. Data Loading & Validation

In [ ]:
DATA_PATH    = 'drive/MyDrive/ML_course/fake_news_full_data.csv'
SAMPLE_FRAC  = 0.2
RANDOM_STATE = 43

full_data = pd.read_csv(DATA_PATH, index_col=0)

print(f'✅ Full dataset loaded: {full_data.shape[0]:,} rows, {full_data.shape[1]} columns')
print(f'   Missing values:\n{full_data.isnull().sum()}')

# Stratified sample to preserve class balance
data = full_data.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
print(f'\n✅ Working sample: {data.shape[0]:,} rows (frac={SAMPLE_FRAC})')

## 2. EDA — Exploratory Data Analysis

### 2.1 Date parsing & cleaning

In [ ]:
# Drop rows with missing dates
before = len(data)
data = data.dropna(subset=['date'])


# Parse dates — coerce invalid to NaT
data['date'] = pd.to_datetime(data['date'], format='mixed', dayfirst=True, errors='coerce')
n_bad_dates = data['date'].isna().sum()
if n_bad_dates:
    print(f'   ⚠️  {n_bad_dates} rows have unparseable dates — dropping them.')
    data = data.dropna(subset=['date'])

print(f'   Date range: {data["date"].min().date()} → {data["date"].max().date()}')

### 2.2 Class balance

In [ ]:
print('📊 Class balance:')
print(data['is_fake'].value_counts(normalize=True).rename({0: 'Real', 1: 'Fake'}).to_string())

print()
for label, name in [(0, 'Real'), (1, 'Fake')]:
    subset = data[data['is_fake'] == label]['date']
    print(f'   {name}: {subset.min().date()} → {subset.max().date()}')

### 2.3 Text length analysis

In [ ]:
data['len_title'] = data['title'].str.len()
data['len_text']  = data['text'].str.len()

print('📐 Length statistics by class:')
display(data.groupby('is_fake')[['len_title', 'len_text']].describe().T)

In [ ]:
stats = data.groupby('is_fake')[['len_title', 'len_text']].describe().T.stack().reset_index()

px.histogram(
    stats[stats['level_0'] == 'len_title'],
    x='level_1', y=0, color='is_fake',
    barmode='group', barnorm='percent',
    title='Title length distribution (%) by class'
).show()

px.histogram(
    stats[stats['level_0'] == 'len_text'],
    x='level_1', y=0, color='is_fake',
    barmode='group', barnorm='percent',
    title='Text length distribution (%) by class'
).show()

In [ ]:
# Very short texts — potential data quality issues
short_texts = data[data['len_text'] <= 10]
print(f'⚠️  Articles with text ≤ 10 chars: {len(short_texts)}')
print(short_texts['is_fake'].value_counts())

# Distribution line charts
lens_text  = data[['len_text',  'is_fake']].value_counts().reset_index().sort_values(['len_text',  'is_fake'])
lens_title = data[['len_title', 'is_fake']].value_counts().reset_index().sort_values(['len_title', 'is_fake'])

px.line(
    lens_text[(lens_text['len_text'] > 43) & (lens_text['len_text'] < 10_000)],
    x='len_text', y='count', facet_row='is_fake',
    title='Article text length distribution'
).show()

px.line(
    lens_title,
    x='len_title', y='count', facet_row='is_fake',
    title='Article title length distribution'
).show()

## 3. Feature Engineering — Date Components

In [ ]:
data['day']       = data['date'].dt.day
data['month']     = data['date'].dt.month
data['year']      = data['date'].dt.year
data['dayofweek'] = data['date'].dt.dayofweek
data = data.drop(columns=['date'])

print('✅ Date features extracted:', ['day', 'month', 'year', 'dayofweek'])

## 4. Train / Test Split

> ⚠️ **Important:** The split happens *before* TF-IDF fitting to prevent data leakage.

In [ ]:
train_idx, test_idx = train_test_split(
    list(data.index), test_size=0.2, shuffle=True,
    stratify=data['is_fake'],   # preserve class ratio
    random_state=RANDOM_STATE
)

train_inputs = data.loc[train_idx].drop('is_fake', axis=1)
train_target = data.loc[train_idx]['is_fake']

test_inputs  = data.loc[test_idx].drop('is_fake', axis=1)
test_target  = data.loc[test_idx]['is_fake']

print(f'✅ Train size: {len(train_inputs):,} | Test size: {len(test_inputs):,}')
print(f'   Train class balance: {train_target.mean():.2%} fake')
print(f'   Test  class balance: {test_target.mean():.2%} fake')

## 5. Text Preprocessing

In [ ]:
english_stopwords = stopwords.words('english')
extra_tokens = ["'d", "'ll", "'m", "'re", "'s", "'ve",
                'could', 'might', 'must', "n't", 'need', 'sha', 'wo', 'would']
english_stopwords.extend(extra_tokens)

lem = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag: str) -> str:
    """Map Penn Treebank POS tag → WordNet POS tag."""
    if treebank_tag.startswith('J'): return wordnet.ADJ
    if treebank_tag.startswith('V'): return wordnet.VERB
    if treebank_tag.startswith('N'): return wordnet.NOUN
    if treebank_tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN  # default


def lemmatization(text: str) -> list:
    """Tokenise → remove stopwords → POS-tag → lemmatise."""
    tokens = [w for w in word_tokenize(text.lower()) if w not in english_stopwords]
    tagged = pos_tag(tokens)
    return [lem.lemmatize(word, get_wordnet_pos(tag)) for word, tag in tagged]


print('✅ Preprocessing functions defined.')

## 6. TF-IDF Vectorisation

In [ ]:
MAX_FEATURES = 1_000

tfidf_text  = TfidfVectorizer(lowercase=True, tokenizer=lemmatization,
                               stop_words=english_stopwords, max_features=MAX_FEATURES)
tfidf_title = TfidfVectorizer(lowercase=True, tokenizer=lemmatization,
                               stop_words=english_stopwords, max_features=MAX_FEATURES)

print('⏳ Fitting TF-IDF on training set (may take a few minutes)…')

# fit on train, transform both — prevents data leakage
text_train  = pd.DataFrame(tfidf_text.fit_transform(train_inputs['text']).toarray(),
                            columns=tfidf_text.get_feature_names_out(),  index=train_inputs.index)
text_test   = pd.DataFrame(tfidf_text.transform(test_inputs['text']).toarray(),
                            columns=tfidf_text.get_feature_names_out(),  index=test_inputs.index)

title_train = pd.DataFrame(tfidf_title.fit_transform(train_inputs['title']).toarray(),
                            columns=tfidf_title.get_feature_names_out(), index=train_inputs.index)
title_test  = pd.DataFrame(tfidf_title.transform(test_inputs['title']).toarray(),
                            columns=tfidf_title.get_feature_names_out(), index=test_inputs.index)

train_set = pd.concat([text_train, title_train], axis=1)
test_set  = pd.concat([text_test,  title_test],  axis=1)

print(f'✅ Feature matrix ready: {train_set.shape[1]} columns')

## 7. Evaluation Helper

In [ ]:
def evaluate_model(model, train_X, test_X, train_y, test_y, model_name: str = ''):
    """
    Fit model on train data, evaluate on test data.
    Prints accuracy, F1, AUC-ROC, confusion matrix, classification report.
    Returns a dict of metrics.
    """
    model.fit(train_X, train_y)
    pred      = model.predict(test_X)
    pred_prob = model.predict_proba(test_X)[:, 1] if hasattr(model, 'predict_proba') else None

    accuracy = accuracy_score(test_y, pred)
    f1       = f1_score(test_y, pred, average='weighted')
    auc      = roc_auc_score(test_y, pred_prob) if pred_prob is not None else None
    cm       = confusion_matrix(test_y, pred)

    sep = '─' * 55
    print(f'{sep}\n  📋 {model_name or model.__class__.__name__}\n{sep}')
    print(f'  Accuracy : {accuracy:.4f}')
    print(f'  F1 (wtd) : {f1:.4f}')
    if auc is not None:
        print(f'  AUC-ROC  : {auc:.4f}')
    print(f'\n  Confusion matrix:\n{cm}')
    print(f'\n{classification_report(test_y, pred, target_names=["Real", "Fake"])}')

    return {'model': model_name, 'accuracy': accuracy, 'f1': f1, 'auc': auc}


results = []   # collect results for final comparison
print('✅ evaluate_model() ready.')

## 8. Logistic Regression

In [ ]:
logreg = LogisticRegression(solver='saga', penalty='l1',
                             max_iter=1_000, random_state=RANDOM_STATE)

res_lr = evaluate_model(logreg, train_set, test_set, train_target, test_target,
                         model_name='Logistic Regression (L1, saga)')
results.append(res_lr)

In [ ]:
coef_df = (
    pd.DataFrame(logreg.coef_[0], index=train_set.columns, columns=['importance'])
    .sort_values('importance', ascending=False)
)

print('🔑 Top-15 features → Fake:')
display(coef_df.head(15))

print('\n🔑 Top-15 features → Real:')
display(coef_df.tail(15))

## 9. Random Forest

In [ ]:
forest = RandomForestClassifier(n_estimators=100, max_depth=30,
                                 random_state=RANDOM_STATE, n_jobs=-1)

res_rf = evaluate_model(forest, train_set, test_set, train_target, test_target,
                         model_name='Random Forest (100 trees, max_depth=30)')
results.append(res_rf)

In [ ]:
feat_imp = (
    pd.Series(forest.feature_importances_, index=train_set.columns)
    .sort_values(ascending=False)
)

print('🌲 Top-20 features by RF importance:')
display(feat_imp.head(20).to_frame('importance'))

px.bar(
    feat_imp.head(30).reset_index(),
    x='index', y='importance',
    title='Top-30 RF feature importances',
    labels={'index': 'Feature'}
).show()

## 10. Hold-out Evaluation (Unseen Data)

In [ ]:
seen_idx     = set(train_idx) | set(test_idx)
holdout_data = full_data.loc[[i for i in full_data.index if i not in seen_idx]].head(2_000)

print(f'🧪 Hold-out sample size: {len(holdout_data)}')
print(f'   Class balance: {holdout_data["is_fake"].mean():.2%} fake')

# Vectorise with already-fitted TF-IDF (no re-fitting!)
holdout_text  = pd.DataFrame(tfidf_text.transform(holdout_data['text']).toarray(),
                              columns=tfidf_text.get_feature_names_out(),
                              index=holdout_data.index)
holdout_title = pd.DataFrame(tfidf_title.transform(holdout_data['title']).toarray(),
                              columns=tfidf_title.get_feature_names_out(),
                              index=holdout_data.index)
holdout_set    = pd.concat([holdout_text, holdout_title], axis=1)
holdout_target = holdout_data['is_fake']

evaluate_model(forest, train_set, holdout_set, train_target, holdout_target,
               model_name='Random Forest — Hold-out')

## 11. BERT Embeddings

> BERT is used here for **embedding extraction only** ([CLS] token → 768-dim vector).  
> To build a full BERT classifier: embed the full train set, then fit a linear head on top.

In [ ]:
BERT_MODEL_NAME = 'bert-base-uncased'
BERT_BATCH_SIZE = 16    # reduce if you hit OOM
MAX_TOKEN_LEN   = 128   # truncate long articles to save memory

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'⚙️  BERT running on: {device}')

bert_tokenizer = BertTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model     = BertModel.from_pretrained(BERT_MODEL_NAME).to(device)
bert_model.eval()
print('✅ BERT model loaded.')

In [ ]:
def get_bert_embeddings(texts: list, batch_size: int = BERT_BATCH_SIZE) -> np.ndarray:
    """
    Returns [CLS] token embeddings (shape: N x 768) for a list of strings.
    Processes in batches to avoid OOM.
    """
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch   = texts[start: start + batch_size]
        encoded = bert_tokenizer(
            batch,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=MAX_TOKEN_LEN
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = bert_model(**encoded)
        cls_emb = output.last_hidden_state[:, 0, :].cpu().numpy()
        all_embeddings.append(cls_emb)
        if (start // batch_size) % 5 == 0:
            print(f'   BERT: {min(start + batch_size, len(texts))}/{len(texts)} texts…')
    return np.vstack(all_embeddings)


print('⏳ Extracting BERT [CLS] embeddings for test set…')
bert_test_emb = get_bert_embeddings(test_inputs['text'].tolist())
print(f'✅ BERT embeddings shape: {bert_test_emb.shape}')

### Next steps for full BERT classifier

```python
# 1. Embed the training set (takes ~10 min on GPU)
bert_train_emb = get_bert_embeddings(train_inputs['text'].tolist())

# 2. Fit a linear head
from sklearn.linear_model import LogisticRegression
bert_clf = LogisticRegression(max_iter=1000)
bert_clf.fit(bert_train_emb, train_target)

# 3. Evaluate
evaluate_model(bert_clf, bert_train_emb, bert_test_emb,
               train_target, test_target, model_name='BERT + LogReg')
```

## 12. Results Summary

In [ ]:
results_df = pd.DataFrame(results).set_index('model')

print('=' * 55)
print('  📊 MODEL COMPARISON')
print('=' * 55)
display(results_df)

px.bar(
    results_df.reset_index().melt(id_vars='model', value_vars=['accuracy', 'f1', 'auc']),
    x='model', y='value', color='variable', barmode='group',
    title='Model comparison: Accuracy / F1 / AUC-ROC',
    range_y=[0.5, 1.0]
).show()